# Data Reading

### Reading CSV Files 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = (spark
      .read
      .format("csv")
      .option("header",True)
      .option("inferSchema",True)
      .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv"))
df.display()

### Reading JSON Files 

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

### Reading Parquet

In [0]:
df_parquet = (spark.
              read
              .format("parquet")
              .load("/Volumes/learnspark/raw/spark_volume/raw_orders/part-00000-tid-orders.c000.snappy.parquet"))
df_parquet.display()

### Reading JDBC

In [0]:
# my_url = "jdbc:postgresql://localhost:5432/postgres"
# myconnection = {"user": "postgres", 
#                 "password": "postgres",
#                 "driver": "org.postgresql.Driver"}
# df = (spark
#       .read
#       .jdbc(url = my_url, table = "orders",properties= myconnection))

# # OR

# df = (
#     spark.read
#     .format("jdbc")
#     .option("url", my_url)
#     .option("dbtable", "orders")
#     .option("user", "postgres")
#     .option("password", "postgres")
#     .option("driver", "org.postgresql.Driver")
#     .load()
# )

# CORRUPT RECORDS MODES

### Permissive 
- reads the complete file, if any corrupt records are they it will be stored in seperate column 

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .option("mode","PREMISSIVE")
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

### DROPMALFORMED 
- it will simple drop malformed record while reading

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .option("mode","DROPMALFORMED")
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

### FAILFAST
- if your downstream application is too sensitive that you cannot take risk, then you can use mode = FAILFAST, so that pipeline will fail immediately 

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .option("mode","FAILFAST")
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

# DATA SCHEMA

### StructType Method

In [0]:
df.schema

In [0]:
my_custom_schme = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_date', DateType(), True), StructField('product_id', StringType(), True), StructField('quantity', IntegerType(), True), StructField('price', DoubleType(), True), StructField('order_status', StringType(), True), StructField('shipping_address', StringType(), True), StructField('city', StringType(), True), StructField('country', StringType(), True), StructField('payment_method', StringType(), True), StructField('discount', DoubleType(), True), StructField('category', StringType(), True), StructField('sales_rep', StringType(), True), StructField('region', StringType(), True), StructField('ship_date', DateType(), True), StructField('delivery_days', IntegerType(), True), StructField('returned', StringType(), True), StructField('gender', StringType(), True)])

In [0]:
df_csv = (spark
          .read
          .format("csv")
          .option("header",True)
          .schema(my_custom_schme)
          .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv"))

df_csv.display()

### DDL Schema

In [0]:
my_ddl_schema = """
order_id INT,
customer_id STRING,
order_date DATE,
product_id STRING,
quantity INTEGER,
price DOUBLE,
order_status STRING,
shipping_address STRING,
city STRING,
country STRING,
payment_method STRING,
discount DOUBLE,
category STRING,
sales_rep STRING,
region STRING,
ship_date DATE,
delivery_days INTEGER,
returned STRING,
gender STRING
"""
df_csv = (spark
          .read
          .format("csv")
          .option("header",True)
          .schema(my_ddl_schema)
          .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv")
         )
display(df_csv)

### SELECT

In [0]:
df_select = df_csv.select("city","country","category")
# OR
df_select = df_csv.select(col("city"),col("country"),col("category"))

df_select.display()

### Alias

In [0]:
df_alias = df_csv.select(col("city").alias("cutomer_city"),col("country").alias("customer_country"),"category")
df_alias.display()

### Filter

#### Scenario_1

In [0]:
display(df_csv.filter(
    col("order_status") == "Returned"
    ))


#### scenario_2

In [0]:
display(df_csv.filter(
    (col("order_status") == "Returned") |
    (col("order_status") == "Cancelled")
    ))

#### isin

In [0]:
desired_order = ["Returned","Cancelled","Shipped"]
display(df_csv.filter(
    col("order_status").isin("Returned","Cancelled","Shipped")
    ))
# OR
display(df_csv.filter(
    col("order_status").isin(desired_order)
    ))

#### withColumnRenamed

In [0]:
df.withColumnRenamed("order_status","status").display()

#### withColumn
- This is the go-to API for either transforming the column or adding/creating a new one

##### scenario_1

In [0]:
df.withColumn("file_path",input_file_name()).withColumn('flag',lit('0')).display()

##### scenario-2

In [0]:
display(
    df.withColumn('shipping_address',regexp_replace("shipping_address",',.*',''))
)

##### scenario-3

In [0]:
display(
    df.withColumn("Total_Price",round(col("price")*col("quantity"),2))
)
# OR
display(
    df.withColumn("Total_Price",col("price")*col("quantity")).withColumn("Total_Price",round("Total_Price",2))
)

### TypeCasting

In [0]:
display(df.withColumn("order_id",col("order_id").cast(StringType())))
# OR
display(df.withColumn("order_id",col("order_id").cast("STRING")))

### Sorting

#### scenario-1

In [0]:
df.sort(col("order_date").desc()).display()

#### scenario-2

In [0]:
display(df.sort(["order_date","quantity"],ascending=[0,1]))

### Limit

In [0]:
display(df.limit(10))

In [0]:
df_drop = df.withColumn("Total_Price",round(col("price")*col("quantity"),2))

### DROP

In [0]:
display(df_drop)
df_drop = df_drop.drop("Total_Price")
display(df_drop)

### dropDuplicates

### scenario-1

In [0]:
df_dedups = df.dropDuplicates()
display(df_dedups)

### scenario-2

In [0]:
df_dedups2 = df.dropDuplicates(subset=["order_date","product_id"])
display(df_dedups2)

### union & unionByName

In [0]:
df_union = df_dedups2.union(df_csv)
display(df_union)

In [0]:
df_change_col_order = df_csv.select("order_date","order_id","customer_id","product_id","quantity","price","order_status","shipping_address","city","country","payment_method","discount","category","sales_rep","region","ship_date","delivery_days","returned","gender"
)

In [0]:
df_union = df_dedups2.union(df_change_col_order)
display(df_union)

In [0]:
df_unionByName = df_dedups2.unionByName(df_change_col_order)
display(df_union)

## Date Functions

In [0]:
df_curr = df.withColumn("current_time",current_timestamp())
display(df_curr)

In [0]:
df_add = df_curr.withColumn("current_time",date_add("current_time",7))
display(df_add)

In [0]:
df_sub = df_curr.withColumn("current_time",date_sub("current_time",7))
display(df_sub)

In [0]:
df_diff = df_curr.withColumn("duration",datediff("current_time","ship_date"))
display(df_diff)

In [0]:
df_fortmat = df_curr.withColumn("current_time",date_format("current_time","yyyy-MM-dd"))
df_fortmat.display()

### STRING FUNCTION

In [0]:
df_upper = df_curr.withColumn("order_status",upper(col("order_status")))
df_upper.display()

In [0]:
df_str = df_curr.withColumn("status_lenght",length("order_status"))
df_str.display()

### Handling Nulls

In [0]:
%py
df_customers = spark.sql("""select * from gizmobox.bronze.v_customers""")

In [0]:
df_customers.display()

In [0]:
df_all_nulls = df_customers.dropna('all')
display(df_all_nulls)

In [0]:
df_any_null = df_customers.dropna('any')
display(df_any_null)

In [0]:
df_subset_null = df_customers.dropna(subset = ["customer_id","email"])
display(df_subset_null)

In [0]:
df_subset_null = df_customers.dropna(subset = ["customer_id","email"],how = 'all') # by default how = 'any'
display(df_subset_null)

In [0]:
df_fillna = df_customers.fillna('dummy')
display(df_fillna)


In [0]:
df_custom_fillna = df_customers.fillna({'email':'dummy@outlook.com','telephone':'+91 9492751026'})
display(df_custom_fillna)


### Split and Indexing

In [0]:
df_split = df_csv.withColumn("street_address",split("shipping_address",",")[0])\
    .withColumn("city_name",split("shipping_address",",")[1])
display(df_split)

### Explode
- it is used to apply explode in the arrays, Exploded in the rows(explode will not keep null, if you need null values to be present use explode_outer)

In [0]:
df_split = df_csv.withColumn("address_array",split("shipping_address",","))
df_explode = df_split.withColumn("address_explode",explode("address_array"))
display(df_explode)

In [0]:
df_split = df_csv.withColumn("address_array",split("shipping_address",","))
df_explode = df_split.withColumn("address_explode",explode_outer("address_array"))
display(df_explode)

### Array Contains

In [0]:
df_arr_contains = df_csv.withColumn("address_array",split("shipping_address",","))\
    .withColumn("city_1_flag",array_contains("address_array"," City1")).select("order_id","address_array","city_1_flag")
display(df_arr_contains)

### Group By

In [0]:
df_agg = (
    df_csv
    .withColumn("total_price",col("price")*col("quantity"))
    .groupBy("product_id")
    .agg(sum("total_price").alias("total_price")).withColumn("total_price",round(col("total_price"),2)).sort(col("total_price").desc())
    .select("product_id","total_price")
    )
display(df_agg)

In [0]:
af_agg_new = (
    df_csv.withColumn("total_sales",round(col("price")*col("quantity"),2))
    .groupBy("product_id","customer_id")
    .agg(sum("total_sales").alias("total_price"),avg("total_sales").alias("avg_price"))
    .withColumn("total_price",round(col("total_price"),2))
    .withColumn("avg_price",round(col("avg_price"),2))
    .sort("product_id","total_price",ascending = [True,False])
)
display(af_agg_new)

### Approx count

In [0]:
df = df_csv.groupBy("product_id").agg(approx_count_distinct("customer_id").alias("distinct_customers"))
display(df)

In [0]:
df_count = df_csv.groupBy("product_id").agg(countDistinct("customer_id").alias("distinct_customers"))
display(df_count)

### Collect_List

In [0]:
df_collect = df_csv.groupBy("customer_id").agg(collect_list('product_id').alias('products'))
display(df_collect)

In [0]:
df_collect = df_csv.groupBy("customer_id").agg(collect_set('product_id').alias('products'))
display(df_collect)